<img src="logo.png" alt="Vegeta" width="240">

# MERLIN — the mission: the race to the fire, the mountain, the smoke (26b)

MERLIN as designed in notebook 26 — read from its export, `designs/data/merlin_design.json` (without it the design file's
defaults) — with its three propulsors read from the propulsor libraries (`designs/data/*_maps.json`, notebooks 25 and 25b):
**nothing is designed or solved again here**; this notebook only flies.

The question: **which propulsor gets MERLIN to a fire 5, 8, 10, 20 or 30 km from the launch site fastest** — the nose
**ducted fan**, the **tractor** or the **pusher** propeller — on the same battery and the same power limit, and still home
with a reserve; then the winner flies the mission through a smoke plume.

```
Part 1  the race:  the three propulsors → thrust and power against speed → reach, return and landing times at 5 / 8 / 10 /
                   20 / 30 km, on flat ground and to a fire 600 m up a mountain → the winner, where the sensor breathes
Part 2  in flight: the winner with its propeller as a rotor disk (CFD, optional) → the mission through a smoke plume
                   → the fire's size from the passes → the movie with the smoke → hand-off
```

`MERLIN_QUICK=1` renders a one-second movie (a check that everything runs).

In [ ]:
import json, math, os, shutil, sys, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display
from vegeta import dedalus, aeromant, core, boreas
from vegeta.aeromant import viz as aviz
sys.path.insert(0, "designs")
import merlin as mdesign
import merlin_flight as mf

ROOT = Path("_runs/merlin_mission"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
OUT = Path("output") / "26b_merlin_mission"; OUT.mkdir(parents=True, exist_ok=True)
RHO, NU, G = 1.225, 1.5e-5, 9.81
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUICK = os.environ.get("MERLIN_QUICK") == "1"
KINDS = ("edf", "tractor", "pusher")
NAME = {"edf": "ducted fan", "tractor": "tractor propeller", "pusher": "pusher propeller"}
COLOR = {"edf": "#d62728", "tractor": "#1f77b4", "pusher": "#ff7f0e"}   # the ducted fan in red

DESIGN = mf.load_design()                     # notebook 26's export (None: the design file's defaults)
MISSION = mf.Mission(**DESIGN["mission"]) if DESIGN else mf.Mission(sampling_agl_m=150.0, sampling_s=180.0, launch_speed=12.0,
                                                                    battery_wh=6 * 3.7 * 8.0, usable_fraction=0.85, reserve_fraction=0.15)
DISTANCES_KM = tuple(DESIGN["distances_km"]) if DESIGN else mf.DISTANCES_KM
MAX_ELECTRICAL_W = DESIGN["max_electrical_w"] if DESIGN else 1900.0
DRIVE_EFF = DESIGN["drive_efficiency"] if DESIGN else 0.85
DESIGN_PARAMS = dict(DESIGN["merlin_params"]) if DESIGN else {}
AOA = 2.0
print("the aircraft:", f"notebook 26's export (wing thickness {DESIGN_PARAMS['thickness']}, CFD corrections {DESIGN['cd0_correction']})"
      if DESIGN else "the design file's defaults (run notebook 26 to export the design)")
pd.DataFrame(DESIGN["variants"]).round(4) if DESIGN else None

## 1. The three propulsors

`mf.build_merlins` puts each propulsor on the airframe of notebook 26's design export (its masses, its polar with the CFD's
correction where it ran) and reads the propulsors from the **propulsor libraries** — nothing is solved here:

- **ducted fan** — notebook 25b's 90 mm, 12-blade, 7-stator fan in MERLIN's duct with the losses fitted to the 3.0 kgf /
  1930 W catalogue point, its map less the annular jet scrubbing the fairing behind the nozzle;
- **tractor** and **pusher** — notebook 25's 10-inch two-blade propeller, the pitch picked from 6, 8 and 10 inches for the
  shortest time to a 20 km fire, its free-stream map at MERLIN's effective wake and thrust deduction.

Notebook 27 races the whole libraries — fans of 70–120 mm with one to three stages, propellers of two to twelve blades,
more power.

In [ ]:
MERLINS = mf.build_merlins(max_electrical_w=MAX_ELECTRICAL_W, drive_efficiency=DRIVE_EFF, mission=MISSION, distance_m=20000.0,
                           design=DESIGN)
info = MERLINS["edf"]["info"]
print(f"EDF ({info['entry']['id']}): duct_loss {info['duct_loss']:.3f}, exit area ratio {info['housing']['exit_area_ratio']:.3f}, "
      f"jet scrubbing {info['scrub_area_m2'] * 1e4:.0f} cm² of fuselage")
for k in ("tractor", "pusher"):
    i = MERLINS[k]["info"]
    print(f"{NAME[k]}: rpm limit {i['rpm_max']:.0f}; chosen {i['prop'].name}, effective wake w = {i['w']:.4f}, thrust deduction t = {i['t']:.4f} (from the library)")
    display(pd.DataFrame(i["pitch_study"]).set_index("pitch [in]").round(4))

## 2. Thrust, power and speed

Full throttle against the aircraft's drag (left): where they cross is the top speed, the gap the acceleration and the
climb. The power in level flight (middle) and the energy per kilometre (right) decide how fast MERLIN can afford to fly
when it must come back. A propeller's thrust falls with airspeed; the ducted fan's small fast jet holds its thrust
longer but pays for it in propulsive efficiency `2V / (V + V_jet)` at every speed.

In [ ]:
PERF = {k: mf.performance(v["airframe"], v["unit"]) for k, v in MERLINS.items()}
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for k, pf in PERF.items():
    ax[0].plot(pf["V"], pf["T_full"], color=COLOR[k], lw=2, label=f"{NAME[k]}: full throttle")
    ax[0].plot(pf["V"], pf["D"], color=COLOR[k], lw=1, ls="--")
    ax[1].plot(pf["V"], pf["P_level"], color=COLOR[k], lw=2, label=NAME[k])
    ax[2].plot(pf["V"], pf["P_level"] / pf["V"] / 3.6, color=COLOR[k], lw=2, label=NAME[k])
ax[0].set(xlabel="airspeed [m/s]", ylabel="N", title="net thrust available (solid) and drag (dashed)", ylim=(0, None)); ax[0].legend(fontsize=8)
ax[1].set(xlabel="airspeed [m/s]", ylabel="W", title="electrical power in level flight", ylim=(0, MAX_ELECTRICAL_W * 1.05)); ax[1].legend(fontsize=8)
ax[2].set(xlabel="airspeed [m/s]", ylabel="Wh/km", title="energy per kilometre", ylim=(0, 4)); ax[2].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({NAME[k]: {"mass [kg]": MERLINS[k]["airframe"].mass_kg, "static thrust [N]": MERLINS[k]["unit"].full(0.0)[0],
                        "top speed [m/s]": pf["v_top"], "best climb [m/s]": pf["roc_max"], "at [m/s]": pf["v_climb"],
                        "best-range speed [m/s]": pf["v_range"], "energy there [Wh/km]": pf["wh_per_km_best"],
                        "stall [m/s]": pf["v_stall"]} for k, pf in PERF.items()}).round(2)

## 3. Time to the fire — 5, 8, 10, 20 and 30 km

`mf.race`: off the stand at 12 m/s, a full-power climb to 150 m at the best-climb speed, a full-power acceleration,
then the dash — at the top speed if the battery allows it, otherwise at the fastest speed that still leaves the energy
for three minutes of sampling, the flight home and a 15 % reserve. Still air. Three times per flight:

- **reach** — launch to overhead the fire: the first reading goes out by radio then, so this is the time that counts;
- **return** — leaving the fire to touchdown: home as fast as the energy left allows (between the best-range speed and the
  top speed), then a 2 m/s approach from 150 m to a belly landing;
- **landing** — launch to touchdown (reach + three minutes of sampling + return): when MERLIN is ready for the next fire.

The reach is won in one of two ways: close in by **thrust** (the top speed), far out by **energy** (the efficiency at
speed). The return gets whatever energy the reach left.

In [ ]:
D_ALL = np.linspace(5.0, 40.0, 36)
RACE = {k: [mf.race(v["airframe"], v["unit"], d * 1000, MISSION) for d in D_ALL] for k, v in MERLINS.items()}
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for k, rr in RACE.items():
    t = [r["time_to_fire_s"] / 60 if r["reachable"] else np.nan for r in rr]
    v = [r["dash_speed"] if r["reachable"] else np.nan for r in rr]
    ax[0].plot(D_ALL, t, color=COLOR[k], lw=2, label=NAME[k]); ax[1].plot(D_ALL, v, color=COLOR[k], lw=2, label=NAME[k])
for d in DISTANCES_KM:
    ax[0].axvline(d, color="#bbbbbb", lw=0.8, zorder=0); ax[1].axvline(d, color="#bbbbbb", lw=0.8, zorder=0)
ax[0].set(xlabel="distance to the fire [km]", ylabel="minutes", title="time from launch to overhead the fire"); ax[0].legend()
ax[1].set(xlabel="distance to the fire [km]", ylabel="m/s", title="dash airspeed it can afford"); ax[1].legend()
RACE_TAB = mf.race_table(MERLINS, DISTANCES_KM, MISSION)
x = np.arange(len(DISTANCES_KM))
for i, k in enumerate(KINDS):
    vals = [RACE_TAB.loc[(f"{d:g} km", NAME[k]), "reach [min]"] for d in DISTANCES_KM]
    ax[2].bar(x + (i - 1) * 0.27, vals, 0.25, color=COLOR[k], label=NAME[k])
    for xx, vv in zip(x + (i - 1) * 0.27, vals):
        ax[2].text(xx, vv, f"{vv:.1f}", ha="center", va="bottom", fontsize=7)
ax[2].set_xticks(x, [f"{d:g} km" for d in DISTANCES_KM]); ax[2].set(ylabel="minutes", title="reach: launch to the fire"); ax[2].legend()
for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()

def timeline(tab, title):
    """Each flight as a bar: reach, sampling, return; the landing at its end."""
    fig, ax = plt.subplots(figsize=(15, 0.42 * len(tab) + 1.2))
    for i, ((dist, name), r) in enumerate(tab.iterrows()):
        k = next(k for k in KINDS if NAME[k] == name)
        ax.barh(i, r["reach [min]"], color=COLOR[k])
        ax.barh(i, MISSION.sampling_s / 60, left=r["reach [min]"], color="#9e9e9e")
        ax.barh(i, r["return [min]"], left=r["reach [min]"] + MISSION.sampling_s / 60, color=COLOR[k], alpha=0.35)
        ax.text(r["landing [min]"] + 0.3, i, f"reach {r['reach [min]']:.1f}  ·  lands {r['landing [min]']:.1f} min", va="center", fontsize=8)
    ax.set_yticks(range(len(tab)), [f"{d}  {n}" for d, n in tab.index]); ax.invert_yaxis()
    ax.set(xlabel="minutes after launch", title=title + " — solid: to the fire, grey: sampling, light: home and landing")
    ax.grid(alpha=0.3, axis="x")
    fig.tight_layout()

timeline(RACE_TAB, "flat forest")
RACE_TAB.round(2)

In [ ]:
WINNER = {}
for d in DISTANCES_KM:
    sub = RACE_TAB.xs(f"{d:g} km", level=0)["reach [min]"].astype(float)
    best = sub.idxmin()
    WINNER[d] = next(k for k in KINDS if NAME[k] == best)
    others = ", ".join(f"{n} +{(t - sub.min()) * 60:.0f} s" for n, t in sub.items() if n != best)
    land = RACE_TAB.xs(f"{d:g} km", level=0)["landing [min]"].astype(float)
    print(f"{d:4g} km: first over the fire: the {best} ({sub.min():.1f} min; {others}); first home: the {land.idxmin()} ({land.min():.1f} min)")
FASTEST = WINNER[20.0]          # the design point: the 20 km case (override here)
print(f"\nMERLIN's propulsion: {NAME[FASTEST]}")

**Reading the race.** The two propellers are within a few percent of each other: the pusher works in the slower
air behind the fuselage and the tail (its effective wake is a gain) and pays a smaller thrust deduction than the tractor,
whose slipstream scrubs the whole fuselage and the wing root. The ducted fan is close only where it can dash flat out
the whole way; past that its energy per kilometre — a small, fast jet — forces it to slow down to keep the way home, and
it falls far behind. What the numbers leave out: the pusher's propeller cuts the tail's wakes (blade vibration and a
louder tone, notebook 25 §10–11), and needs the belly landing to keep the blades off the ground (a folding propeller);
the EDF's protected rotor and its tolerance of smoke and debris.

## 3b. A fire on a mountain

The same race to a fire **600 m above the launch site** (a hillside fire; the launch in the valley). MERLIN climbs to its
150 m sampling height over the launch site as before, then climbs gently **all along the dash** — the angle that gains the
600 m over the remaining distance, 7° at 5 km, 1.2° at 30 km. The thrust then also carries the weight's share along the
path, `W sin γ`: the top speed falls and the energy rises by `m g Δh`. The flight home descends the same 600 m and gets
part of it back (less thrust, a glide where the descent pays for all the drag). The air at 900 m is about 6 % thinner
(less drag, less thrust): not modelled — the slope is the effect shown here.

In [ ]:
FIRE_ELEVATION_M = 600.0
MOUNTAIN_TAB = mf.race_table(MERLINS, DISTANCES_KM, MISSION, fire_elevation_m=FIRE_ELEVATION_M)
timeline(MOUNTAIN_TAB, f"fire {FIRE_ELEVATION_M:.0f} m up a mountain")
cmp_ = pd.DataFrame({("reach [min]", "flat"): RACE_TAB["reach [min]"], ("reach [min]", "mountain"): MOUNTAIN_TAB["reach [min]"],
                     ("landing [min]", "flat"): RACE_TAB["landing [min]"], ("landing [min]", "mountain"): MOUNTAIN_TAB["landing [min]"],
                     ("dash [m/s]", "flat"): RACE_TAB["dash [m/s]"], ("dash [m/s]", "mountain"): MOUNTAIN_TAB["dash [m/s]"],
                     ("dash", "climb [deg]"): MOUNTAIN_TAB["dash climb [deg]"]}).astype(float)
cmp_[("reach [min]", "uphill costs [s]")] = (cmp_[("reach [min]", "mountain")] - cmp_[("reach [min]", "flat")]) * 60
for d in DISTANCES_KM:
    sub = MOUNTAIN_TAB.xs(f"{d:g} km", level=0)["reach [min]"].astype(float)
    print(f"{d:4g} km uphill: first over the fire: the {sub.idxmin()} ({sub.min():.1f} min)")
cmp_.round(2)

## 4. Where the sensor breathes

The gas sensor must sample air the propulsor has not touched. On the **pusher** the nose is clean. On the **tractor**
the nose is the propeller: the sensor sits in a wing-root intake, inside the slipstream, which mixes the plume's
structure. The **EDF**'s inlet swallows a stream tube several times its own area at low speed — a sensor port in the lip
reads the air being drawn in, not the free stream ahead.

In [ ]:
e_fan = MERLINS["edf"]["info"]["entry"]                    # the library map: jet speed over (V, rpm)
A_exit = e_fan["exit_area_ratio"] * e_fan["fan_area_m2"]
A_high = math.pi * (MERLINS["edf"]["info"]["housing"]["highlight_radius"] / 1000) ** 2
A_prop = math.pi * (MERLINS["tractor"]["info"]["prop"].diameter / 2) ** 2
rows = {}
for V in (15.0, 30.0, 50.0):
    rpm = MERLINS["edf"]["unit"].full(V)[2]
    vj = float(np.interp(rpm, e_fan["rpm"], [np.interp(V, e_fan["V"], e_fan["exit_velocity"][:, j]) for j in range(len(e_fan["rpm"]))]))
    T = MERLINS["tractor"]["unit"].full(V)[0]                 # far-wake excess from momentum: T = rho A (V + u) u, u = jet - V
    u = math.sqrt(V ** 2 + 2 * T / (RHO * A_prop)) - V
    rows[f"{V:.0f} m/s"] = {"EDF capture area / highlight area": A_exit * vj / V / A_high, "EDF jet speed [m/s]": vj,
                            "tractor slipstream excess, far wake [m/s]": u}
pd.DataFrame(rows).round(2)

# Part 2 — the winner in flight

## 5. The whole aircraft with its propeller (Aeromant, optional)

The winner's airframe at 2° with its propeller as one rotor disk (`hull_rotor_disk`: the single-disk case of
`aircraft_rotor_disks`, blade-element source terms from the Boreas blade and polar) at the 20 km race's dash speed,
against the airframe alone. Only for a propeller: the EDF's rotor sits inside its duct, a case for `rotor_mrf_installed`
(notebook 25 §13.5).

In [ ]:
cfd_disk = None
if RUN_CFD and FASTEST != "edf":
    prop = MERLINS[FASTEST]["info"]["prop"]
    r20 = mf.race(MERLINS[FASTEST]["airframe"], MERLINS[FASTEST]["unit"], 20000, MISSION)
    rpm = MERLINS[FASTEST]["unit"].full(r20["dash_speed"])[2]
    design_file = ROOT / "merlin.py"
    shutil.copy(Path("designs/merlin.py"), design_file)
    ws = core.Workspace.create(ROOT / "workspace", name="MERLIN mission")
    mdes = ws.add_design("merlin", f"{design_file}:Merlin")
    rev = mdes.new_revision(note=f"{NAME[FASTEST]} with its rotor disk",
                            **dict(DESIGN_PARAMS, part="aircraft", propulsion=FASTEST, angle_of_attack_deg=AOA))
    rev.generate(stl_tolerance=0.2)
    th = math.radians(AOA)
    lay = mdesign.Merlin.layout(rev.params)
    x, z = lay["prop_x"], lay["axis_z"]
    centre = [(x * math.cos(th) + z * math.sin(th)) / 1000, 0.0, (-x * math.sin(th) + z * math.cos(th)) / 1000]
    axis = [-math.cos(th), 0.0, math.sin(th)]
    sec = boreas.Airfoil(**mf.SECTION_KW)
    alphas = np.unique(np.r_[np.arange(-180, -30, 10), np.arange(-30, 31, 1), np.arange(40, 181, 10)]).astype(float)
    cl_, cd_ = sec.coefficients(np.radians(alphas))
    disk = aeromant.CFDCase("hull_rotor_disk", rev.stl,
        dict(velocity=r20["dash_speed"], kinematic_viscosity=NU, density=RHO, reference_area=MERLINS[FASTEST]['airframe'].wing_area_m2, reference_length=DESIGN_PARAMS.get("span", mdesign.Merlin().resolve()["span"]) / 6000,
             center_of_rotation=(0.07, 0.0, 0.0), iterations=600, residual_target=1e-4, surface_level=4, near_level=3, wake_level=2,
             cells_per_length=2.0, disk1_center=centre, disk_axis=axis, diameter=prop.diameter, rpm=rpm, blades=prop.blades,
             blade=[[r_, b_, c_] for r_, b_, c_ in zip(prop.r, prop.beta_deg, prop.chord)],
             polar=[[a_, float(d_), float(l_)] for a_, l_, d_ in zip(alphas, cl_, cd_)], rotation1=1, disk_level=5),
        workdir=ROOT / "cfd_disk", geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    print(disk.prepare(overwrite=True))
    cfd_disk = disk.run(progress=True)
    print(cfd_disk)
    if cfd_disk.ok:
        aviz.show(aviz.plot_field_slice(disk, "U", normal="y", origin=(0, 0.0, 0)))
        clip = aviz.animate_particles(disk, OUT / f"merlin_{FASTEST}_flow.mp4", n=500, seconds=6, fps=24)
        display(Video(str(clip), embed=False, width=900))
else:
    print("rotor-disk CFD skipped" + (" (VEGETA_SKIP_OPENFOAM=1)" if not RUN_CFD else " (the winner is the ducted fan)"))

## 6. The mission through the smoke

A 40 MW fire (a front of a few tens of metres in a forest) 20 km east of the launch site, a 5 m/s south-west wind
carrying the column north-east. The smoke is a Gaussian plume (`mf.Plume`: Briggs buoyant rise, Pasquill–Gifford
class C dispersion, CO at 0.10 kg per kg of fuel). MERLIN dashes at the race's speed, flies crosswind passes 600 m
downwind at four heights around 150 m, and goes home at the best-range speed into the wind. The sensor has a one-second
time constant.

In [ ]:
PLUME = mf.Plume(source_xy=(20000.0, 0.0), heat_mw=40.0, wind_speed=5.0, wind_from_deg=225.0)
FOREST = mf.Forest()
af, unit = MERLINS[FASTEST]["airframe"], MERLINS[FASTEST]["unit"]
R20 = mf.race(af, unit, 20000.0, MISSION)
EP = mf.fly(af, unit, FOREST, PLUME, race_result=R20, mission=MISSION)
print(json.dumps({k: v for k, v in EP.summary().items()}, default=float, indent=1))
display(EP.phase_table().round(1))
fig = mf.profile_figure(EP, PLUME)

### Is it real, and how big?

Each pass integrates the CO across the column at its height; a Gaussian with ground reflection through the four
integrals gives the CO flux, the plume's height and its depth — and with the emission factor and the fuel's heat of
combustion, the fire's heat release. The model's own fire is 40 MW: the retrieval is checked against the truth here; in
the field the wind speed is measured on board.

In [ ]:
EST = mf.source_estimate(PLUME, EP.passes)
zs = np.linspace(0, 400, 200)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for p_ in EP.passes:
    s_, c_ = PLUME.local(p_["x"], p_["y"])
    ax[0].plot(c_, p_["ppm"], lw=1.5, label=f"{p_['name']} at {np.mean(p_['z_agl']):.0f} m")
ax[0].set(xlabel="crosswind position [m]", ylabel="CO above background [ppm]", title="the passes"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot(EST["cwic_kg_m2"] * 1e6, EST["pass_heights_m"], "o", color="k", label="passes")
ax[1].plot(PLUME.cwic(EST["downwind_m"], zs, EST["co_kg_s"], EST["plume_height_m"], EST["sigma_z_m"]) * 1e6, zs, color=COLOR[FASTEST], label="fit")
ax[1].plot(PLUME.cwic(EST["downwind_m"], zs) * 1e6, zs, ":", color="#555555", label="the model's truth")
ax[1].set(xlabel="crosswind-integrated CO [mg/m²]", ylabel="height AGL [m]", title="through the column"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.Series({"peak CO [ppm]": EST["peak_ppm"], "CO flux [g/s]": EST["co_kg_s"] * 1000, "true CO flux [g/s]": PLUME.co_kg_s * 1000,
           "plume height [m]": EST["plume_height_m"], "fire heat release [MW]": EST["heat_mw"], "true [MW]": PLUME.heat_mw,
           "confirmed": EST["peak_ppm"] > 10 * 0.05}).to_frame("ignition check")

## 7. The movie — with the smoke

The forest around the aircraft, the smoke column as particles drawn from the plume model (darker and denser near the
fire, paler and wider downwind), MERLIN and its trail, the map and the sensor's reading. Time-lapse: half the movie is
the passes through the smoke.

In [ ]:
MOVIE, FRAMES = mf.render_movie(EP, FOREST, PLUME, OUT / f"merlin_{FASTEST}_smoke_20km.mp4", seconds=1.0 if QUICK else 24.0,
                                fps=10 if QUICK else 20, title=f"MERLIN ({NAME[FASTEST]}) — a fire 20 km out", progress=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 6.2))
for a, fr in zip(axes.flat, np.linspace(0, len(FRAMES) - 1, 6).astype(int)):
    a.imshow(FRAMES[fr][:, :, ::-1]); a.axis("off")
fig.tight_layout()
Video(str(MOVIE), embed=False, width=960)

## 8. Hand-off

In [ ]:
handoff = {
    "source": "26b_merlin_mission", "design": "notebook 26's export" if DESIGN else "design file defaults", "propulsion": FASTEST, "distances_km": list(DISTANCES_KM),
    "winner_per_distance": {f"{d:.0f} km": WINNER[d] for d in DISTANCES_KM},
    "race": {f"{i[0]} / {i[1]}": {k: (v if not isinstance(v, (np.floating,)) else float(v)) for k, v in r.items()} for i, r in RACE_TAB.iterrows()},
    "race_mountain": {"fire_elevation_m": FIRE_ELEVATION_M, **{f"{i[0]} / {i[1]}": {k: (v if not isinstance(v, (np.floating,)) else float(v)) for k, v in r.items()} for i, r in MOUNTAIN_TAB.iterrows()}},
    "masses_kg": {NAME[k]: MERLINS[k]["airframe"].mass_kg for k in KINDS},
    "polar": {NAME[k]: {"cd0": MERLINS[k]["airframe"].cd0, "S": MERLINS[k]["airframe"].wing_area_m2, "AR": MERLINS[k]["airframe"].aspect_ratio} for k in KINDS},
    "mission": EP.summary(), "source_estimate": {k: v for k, v in EST.items() if not isinstance(v, np.ndarray)},
    "preferred_wing": DESIGN.get("preferred_wing") if DESIGN else None,
    "movie": str(MOVIE),
}
Path("_runs/merlin_mission_handoff.json").write_text(json.dumps(handoff, indent=1, default=str))
print("written _runs/merlin_mission_handoff.json")

**Next steps:** a folding pusher propeller and its blade under the tail's wakes (notebook 25's pusher sections on this
propeller); the race in a head- and tailwind; the sensor's inlet in CFD; a wind estimate on board for the flux; the
battery's voltage sag at 12 C; the landing skid and the belly-landing load case.